In [1]:
import numpy as np
import matplotlib.pyplot as plt
import awkward as ak
import uproot
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
from utils import *

%load_ext autoreload
%autoreload 2

# 1. Data loading

In [2]:
ES_ROOT  = "/home/yujin/projects/wind/WIND_bkg_rejection/ROOT_FILEs/WIND_66_4in_40p_ES_10k_internal_PMT.ntuple.root"
# 16N_ROOT = "/home/yujin/projects/wind/BKG_rejection/dataset/WIND_66_4in_40p_16N_10k_internal.ntuple.root"

branches = ["mcPEx", "mcPEy", "mcPEz", "mcPECharge", "mcPEHitTime"]
temp_branches = ["mcid", "mcke", "mcnhits", "mcx", "mcy", "mcz"] 
label = 1

In [3]:

with uproot.open(f"{ES_ROOT}:output") as tree:
            full_branches = branches + temp_branches
            full_data = tree.arrays(full_branches, library="ak")

print(f"Load from '{ES_ROOT}'")
print(f" - # of event in the root file      : {len(full_data)}")

# # Evenet Selection
# if label == 1: # ES
#     data = es_event_selection(full_data, energy_thr, vtx_cut_mm, min_unique_pmt)
# else:
#     data = bkg_event_selection(full_data, energy_thr, vtx_cut_mm, min_unique_pmt)

# print(f" - # of event after event selection : {len(data)}")

Load from '/home/yujin/projects/wind/WIND_bkg_rejection/ROOT_FILEs/WIND_66_4in_40p_ES_10k_internal_PMT.ntuple.root'
 - # of event in the root file      : 10000


In [4]:
# 2D Jagged array
full_data

<Array [{mcPEx: [...], mcPEy: [...], ...}, ...] type='10000 * {mcPEx: var *...'>

In [5]:
mcid = full_data["mcid"]
mcid

<Array [0, 1, 2, 3, 4, ..., 9995, 9996, 9997, 9998, 9999] type='10000 * int32'>

In [6]:
inx = 3
event = full_data[inx]
event

<Record {mcPEx: [...], mcPEy: [...], ...} type='{mcPEx: var * float64, mcPE...'>

In [7]:
# 1D jagged array
x_pos = event["mcPEx"]
y_pos = event["mcPEy"]
z_pos = event["mcPEz"]
charge = event["mcPECharge"]
hit_time = event["mcPEHitTime"]
x_pos

<Array [2.26e+03, 2.26e+03, 1.32e+03, ..., 375, 780, -450] type='118 * float64'>

# 2. PMT Info

In [ ]:
pmt_types = classify_pmt_type(x_pos,y_pos,z_pos)
pmt_types

<Array [0, 0, 0, 1, 2, 0, 2, 2, ..., 0, 0, 0, 0, 1, 0, 0, 2] type='118 * int64'>

In [ ]:
unique_pmts = unique_pmt_count_per_event(x_pos, y_pos, z_pos)
unique_pmts

np.int64(93)

# 3. Evnet Selection

In [10]:
np_mcid = ak.to_numpy(mcid)
mask, max_id = build_eventid_mask(np_mcid)
mask

array([ True,  True,  True, ...,  True,  True,  True], shape=(10000,))

In [11]:
m, mx, size = get_selected_es_eventids_mask(ES_ROOT, "kinetic", 0)
m

 - Energy cut: thr=0 MeV, criteria=kinetic
     Input E  : Total energy
     Selected : 8700 / 10000 (87.00%)


array([ True,  True,  True, ...,  True,  True,  True], shape=(10000,))

In [13]:
get_gen_vertex_mask_cylinder(ES_ROOT, 3000, 3000, 0)

 - Vertex cut (Cylinder): Wall margin = 0 mm
     Fiducial R/Z    : R < 3000.0 mm, |Z| < 3000.0 mm
     Fiducial Volume : 169.646 m^3
     Fiducial Mass   : 169.646 tons (as Water)
     Selected        : 10000 / 10000 (100.00%)


(array([ True,  True,  True, ...,  True,  True,  True], shape=(10000,)),
 9999,
 10000,
 10000)

In [ ]:
es_event_selection(ES_ROOT, full_data, 0, 0, 0)

# 4. Channel Maps

## 1.3 Visualizations

In [ ]:
def hitmap_2d(hits_x, hits_y, hits_z, hits_values):
    if not (len(hits_x) == len(hits_y) == len(hits_z) == len(hits_values)):
        print("Error: Input arrays have different lengths")
        return
    
    n_hits = len(hits_x)
    data = {i: {k: [] for k in ['phi', 'phi_deg', 'r', 'z', 'v']} for i in range(3)}
    p_types = classify_pmt_type_ak_batch(hits_x, hits_y, hits_z)

    for i in range(n_hits):
        this_type = p_types[i]
        phi_rad = np.arctan2(hits_y[i], hits_x[i]) % (2 * np.pi)
        phi_deg = np.degrees(phi_rad)
        r = np.sqrt(hits_x[i]**2 + hits_y[i]**2)
        
        data[this_type]['phi'].append(phi_rad)
        data[this_type]['phi_deg'].append(phi_deg)
        data[this_type]['r'].append(r)
        data[this_type]['z'].append(hits_z[i])
        data[this_type]['v'].append(hits_values[i])

    v_min, v_max = (np.min(hits_values), np.max(hits_values)) if n_hits > 0 else (0, 1)

    fig = plt.figure(figsize=(16, 22))
    gs = gridspec.GridSpec(3, 1, height_ratios=[1.2, 0.8, 1.2], hspace=0.3)
    cmap = 'viridis'

    # --- (1) Top ---
    ax0 = fig.add_subplot(gs[0], projection='polar')
    ax0.set_theta_zero_location("N")
    ax0.set_theta_direction(1) # 반시계 방향
    sc0 = ax0.scatter(data[0]['phi'], data[0]['r'], c=data[0]['v'], 
                     cmap=cmap, vmin=v_min, vmax=v_max, s=30, alpha=0.8)
    ax0.set_ylim(0, TANK_RADIUS)
    ax0.set_title(f"Top (Hits: {len(data[0]['phi'])})", fontsize=16, pad=30)

    # --- (2) Side (Mantle) ---
    ax2 = fig.add_subplot(gs[1])
    sc2 = ax2.scatter(data[2]['phi_deg'], data[2]['z'], c=data[2]['v'], 
                     cmap=cmap, vmin=v_min, vmax=v_max, s=20, alpha=0.6)
    degree_ticks = np.arange(0, 361, 45)
    ax2.set_xticks(degree_ticks)
    ax2.set_xticklabels([f'{d}°' for d in degree_ticks])
    ax2.set_xlim(0, 360)
    ax2.set_ylim(Z_BOT - 10, Z_TOP + 10)
    ax2.set_title(f"Side (Hits: {len(data[2]['phi_deg'])})", fontsize=16)
    plt.colorbar(sc2, ax=ax2, fraction=0.015, pad=0.04)

    # --- (3) Bottom: 
    ax1 = fig.add_subplot(gs[2], projection='polar')
    ax1.set_theta_zero_location("S") 
    ax1.set_theta_direction(-1) 
    sc1 = ax1.scatter(data[1]['phi'], data[1]['r'], c=data[1]['v'], 
                     cmap=cmap, vmin=v_min, vmax=v_max, s=30, alpha=0.8)
    ax1.set_ylim(0, TANK_RADIUS)
    ax1.set_title(f"Bottom (Hits: {len(data[1]['phi'])})", fontsize=16, pad=30)

    plt.subplots_adjust(left=0.1, right=0.9, top=0.95, bottom=0.05)
    plt.show()

    # 히트 통계 출력 복구
    print("-" * 30)
    print(f"Stats - Top: {len(data[0]['phi'])}, Side: {len(data[2]['phi_deg'])}, Bottom: {len(data[1]['phi'])}")
    print(f"Total Hits Processed: {n_hits}")
    print("-" * 30)

In [ ]:
def hitmap_3d(hits_x, hits_y, hits_z, hits_values):
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # 1. 배경 및 격자 제거
    ax.grid(False) # 격자 제거
    ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0)) # X축 배경 투명화
    ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0)) # Y축 배경 투명화
    ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0)) # Z축 배경 투명화

    # 2. 히트 데이터 플롯
    v_min, v_max = (np.min(hits_values), np.max(hits_values)) if len(hits_values) > 0 else (0, 1)
    sc = ax.scatter(hits_x, hits_y, hits_z, c=hits_values, 
                    cmap='viridis', vmin=v_min, vmax=v_max, s=25, alpha=0.9)

    # 3. 검출기 기하학적 구조 가이드
    theta = np.linspace(0, 2*np.pi, 100)
    ax.plot(TANK_RADIUS * np.cos(theta), TANK_RADIUS * np.sin(theta), Z_TOP, color='red', linestyle='--', linewidth=2)
    ax.plot(TANK_RADIUS * np.cos(theta), TANK_RADIUS * np.sin(theta), Z_BOT, color='red', linestyle='--', linewidth=2)

    z_side = np.linspace(Z_BOT, Z_TOP, 10)
    theta_side = np.linspace(0, 2*np.pi, 30)
    theta_grid, z_grid = np.meshgrid(theta_side, z_side)
    ax.plot_wireframe(TANK_RADIUS * np.cos(theta_grid), TANK_RADIUS * np.sin(theta_grid), z_grid, 
                      color='black', alpha=0.2, linewidth=0.5)

    ax.set_xlabel('X [mm]')
    ax.set_ylabel('Y [mm]')
    ax.set_zlabel('Z [mm]')
    
    plt.colorbar(sc, ax=ax, fraction=0.02, pad=0.1)
    plt.show()

## 1.4 Data Preprocessing (CHW Array)

In [ ]:
def to_chw_arr(x, y, z, channels=[], width=142, side_height=45, cap_res=23):
    # z축 및 반지름 고정 범위
    z_min, z_max, r_max = -3000.0, 3000.0, 3000.0
    n_channels = len(channels)
    
    # 전체 높이 계산: 하단캡 + 사이드(45) + 상단캡
    total_height = side_height + (2 * cap_res) # 11 + 45 + 11 = 67
    
    chw = np.zeros((n_channels, total_height, width))
    p_types = classify_pmt_type_ak_batch(x, y, z)

    # 가로축 Phi 인덱스 (0 ~ 141)
    phi = np.arctan2(y, x) % (2 * np.pi)
    phi_idx = (phi / (2 * np.pi) * (width - 1)).astype(int)
    phi_idx = np.clip(phi_idx, 0, width - 1)

    for c_idx in range(n_channels):
        values = channels[c_idx]

        for i in range(len(x)):
            this_type = p_types[i]
            pi = phi_idx[i]
            r = np.sqrt(x[i]**2 + y[i]**2)
            
            if this_type == 2:  # Side
                # 사이드는 인덱스 cap_res(11)부터 시작해서 side_height(45)만큼 점유
                z_side_norm = (z[i] - z_min) / (z_max - z_min)
                zi = cap_res + int(z_side_norm * (side_height - 1))
                zi = np.clip(zi, cap_res, cap_res + side_height - 1)
                chw[c_idx, zi, pi] += values[i]
                
            elif this_type == 0: # Top Cap
                # 상단 영역: 사이드 끝 인덱스 위로 r에 따라 배치
                ri = int((r / r_max) * (cap_res - 1))
                ri = np.clip(ri, 0, cap_res - 1)
                # r=r_max(사이드 인접) -> zi = cap_res + side_height
                # r=0(중심) -> zi = total_height - 1
                zi = (total_height - 1) - (cap_res - 1 - ri) 
                chw[c_idx, zi, pi] += values[i]
                
            elif this_type == 1: # Bottom Cap
                # 하단 영역: 0부터 r에 따라 배치
                ri = int((r / r_max) * (cap_res - 1))
                ri = np.clip(ri, 0, cap_res - 1)
                # r=r_max(사이드 인접) -> zi = cap_res - 1
                # r=0(중심) -> zi = 0
                zi = ri 
                chw[c_idx, zi, pi] += values[i]
                
    return chw

In [ ]:
def get_geometry_channels(width=142, side_height=45, cap_res=23):
    """
    검출기 기하학 구조를 반영한 2채널 NumPy 배열 생성
    - Channel 0: Z-axis position (Normalized -1 to 1)
    - Channel 1: Radial/XY position (Normalized 0 to 1)
    
    Returns:
        np.ndarray: Shape (2, 91, 142), dtype=float32
    """
    total_height = side_height + (2 * cap_res) # 23 + 45 + 23 = 91
    
    # (2, 91, 142) 크기의 빈 배열 생성
    geo_map = np.zeros((2, total_height, width), dtype=np.float32)
    
    # --- 채널 0: Z-Position Correction ---
    # 전체 91개 행에 대해 하단(-1)부터 상단(1)까지 선형 배분
    z_values = np.linspace(-1, 1, total_height)
    for zi in range(total_height):
        geo_map[0, zi, :] = z_values[zi]
        
    # --- 채널 1: XY (Radial) Position Correction ---
    # Cap은 중심(0)~외곽(1) 매핑, Side는 외곽(1) 고정
    for zi in range(total_height):
        if zi < cap_res:  # Bottom Cap 영역
            r_val = zi / (cap_res - 1)
        elif zi >= cap_res + side_height:  # Top Cap 영역
            r_val = (total_height - 1 - zi) / (cap_res - 1)
        else:  # Side 영역
            r_val = 1.0
            
        geo_map[1, zi, :] = r_val
        
    return geo_map

In [ ]:
def plot_chw(chw, cap_res=23):
    n_channels, height, width = chw.shape
    
    channel_names = [f"Channel {i}" for i in range(n_channels)]
    
    # 채널 수만큼 서브플롯 생성
    fig, axes = plt.subplots(n_channels, 1, figsize=(15, 5 * n_channels))
    
    # 채널이 1개일 경우 axes가 배열이 아니므로 처리
    if n_channels == 1:
        axes = [axes]
        
    for i in range(n_channels):
        # aspect='auto'로 가로세로 비율 최적화, origin='lower'로 물리적 방향 일치
        im = axes[i].imshow(chw[i], aspect='auto', cmap='viridis', origin='lower')
        
        # --- 영역 구분 점선 추가 ---
        # 1. Bottom Cap과 Side 사이의 경계
        axes[i].axhline(y=cap_res - 0.5, color='white', linestyle='--', linewidth=1.5, alpha=0.8)
        # 2. Side와 Top Cap 사이의 경계 (전체 높이에서 cap_res만큼 내려온 지점)
        axes[i].axhline(y=height - cap_res - 0.5, color='white', linestyle='--', linewidth=1.5, alpha=0.8)
        
        # 텍스트 라벨 추가 (선택 사항)
        axes[i].text(5, cap_res/2, 'Bottom', color='white', verticalalignment='center', fontweight='bold')
        axes[i].text(5, height/2, 'Side', color='white', verticalalignment='center', fontweight='bold')
        axes[i].text(5, height - cap_res/2, 'Top', color='white', verticalalignment='center', fontweight='bold')

        axes[i].set_title(channel_names[i], fontsize=15)
        axes[i].set_xlabel("Phi Index")
        axes[i].set_ylabel("Z Index")
        
        plt.colorbar(im, ax=axes[i], label='Intensity')

    plt.tight_layout()
    plt.show()

# 3. 

## 2.2 2D Hitmap

In [ ]:
event_id = 3
%matplotlib inline
hitmap_2d(x_arr[event_id], y_arr[event_id], z_arr[event_id], t_arr[event_id])

In [ ]:
# %matplotlib tk
hitmap_3d(x_arr[event_id], y_arr[event_id], z_arr[event_id], t_arr[event_id])

In [ ]:
event_id = 3
chw = to_chw_arr(x_arr[event_id], y_arr[event_id], z_arr[event_id], [t_arr[event_id]], 142, 45, 23)
plot_chw(chw, 23)

In [ ]:
geo = get_geometry_channels(142,45,23)
plot_chw(geo,23)

In [ ]:
img = get_channels(x_arr[event_id], y_arr[event_id], z_arr[event_id], t_arr[event_id], c_arr[event_id])
plot_chw(img)